In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/awaispasha/macro-analysis/real_interest_rate_by_country.csv
/kaggle/input/datasets/awaispasha/macro-analysis/world-data-2023.csv
/kaggle/input/datasets/awaispasha/macro-analysis/unemployment_analysis.csv
/kaggle/input/datasets/awaispasha/macro-analysis/world_gdp_data.csv
/kaggle/input/datasets/awaispasha/macro-analysis/exchange_rates.csv


In [2]:
pd.set_option("display.max_columns", None)


# 1. Introduction

Understanding how monetary conditions affect labor markets is a central question in macroeconomics. Policymakers frequently adjust interest rates with the expectation that tighter or looser monetary conditions will influence employment, economic activity, and overall welfare. However, empirical evidence on the relationship between interest rates and unemployment—especially across countries and during periods of economic crisis—remains mixed.

This project investigates the relationship between real interest rates and unemployment using a global panel dataset spanning multiple countries and years. Rather than focusing on a single economy, the analysis adopts a comparative international perspective, allowing for structural heterogeneity across labor markets and policy regimes. In addition, the project explicitly examines whether global crises, such as the 2008 Global Financial Crisis and the 2020 COVID‑19 pandemic, alter the relationship between monetary conditions and unemployment.

To address these questions, the study combines descriptive analysis, panel regression techniques, lagged specifications, and crisis interaction models. The goal is not only to estimate statistical relationships, but also to develop an interpretable and policy‑relevant narrative about unemployment dynamics in a global context.

In [3]:
BASE_PATH = "/kaggle/input/datasets/awaispasha/macro-analysis/"

unemp = pd.read_csv(BASE_PATH + "unemployment_analysis.csv")
rir = pd.read_csv(BASE_PATH + "real_interest_rate_by_country.csv")

exrate = pd.read_csv(BASE_PATH + "exchange_rates.csv")
world_data = pd.read_csv(BASE_PATH + "world-data-2023.csv")

world_gdp = pd.read_csv(
    BASE_PATH + "world_gdp_data.csv",
    encoding="latin1"
)

# 2. Data Preparation and Methodology

# 2.1 Data Sources and Cleaning
(Corresponds to: “Clean Unemployment Data”, “Clean Real Interest Rate Data”, “Clean Remaining Datasets”)
The analysis integrates multiple macroeconomic datasets, including unemployment rates, real interest rates, GDP, and auxiliary country‑level indicators. Because these datasets originate from different sources and formats, significant preprocessing was required.

Unemployment data were initially provided in a wide country‑year format and were reshaped into a long panel structure, enabling time‑series and panel analysis. Real interest rate data were cleaned and standardized to ensure consistent country identifiers and numeric formats. Additional datasets—such as GDP and country‑level indicators—were cleaned separately, with attention to differences between panel and cross‑sectional data structures.

The result of this process is a **clean, harmonized country‑year panel dataset**, suitable for econometric analysis.

# 2.2 Empirical Strategy

The empirical approach follows a **layered strategy**:

1. **Exploratory analysis** to understand global trends and heterogeneity
2. **Pooled regressions** to establish baseline correlations
3. **Fixed‑effects models** to control for unobserved country and time heterogeneity
4. **Lagged specifications** to test delayed monetary transmission
5. **Crisis and interaction models** to capture structural breaks during global shocks

This progression mirrors best practices in applied macroeconomics and ensures that results are robust to alternative specifications.

# Clean Unemployment Data (Wide → Long)

In [4]:
year_cols = [c for c in unemp.columns if c.isdigit()]

unemp_long = unemp.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="unemployment_rate"
)

unemp_long.columns = ["country", "country_code", "year", "unemployment_rate"]
unemp_long["year"] = unemp_long["year"].astype(int)
unemp_long["unemployment_rate"] = pd.to_numeric(
    unemp_long["unemployment_rate"], errors="coerce"
)

unemp_long = unemp_long.dropna(subset=["unemployment_rate"])

# Clean Real Interest Rate Data

In [5]:
rir = rir.rename(columns={
    "country_name": "country",
    "value": "real_interest_rate"
})

rir["year"] = rir["year"].astype(int)
rir["real_interest_rate"] = pd.to_numeric(
    rir["real_interest_rate"], errors="coerce"
)

rir = rir.dropna(subset=["real_interest_rate"])

# Clean Remaining Datasets

In [6]:
for name, df in {
    "exrate": exrate,
    "world_data": world_data,
    "world_gdp": world_gdp
}.items():
    print(f"\n{name} columns:")
    print(df.columns.tolist())


exrate columns:
['Unnamed: 0', 'Country/Currency', 'currency', 'value', 'date']

world_data columns:
['Country', 'Density\n(P/Km2)', 'Abbreviation', 'Agricultural Land( %)', 'Land Area(Km2)', 'Armed Forces size', 'Birth Rate', 'Calling Code', 'Capital/Major City', 'Co2-Emissions', 'CPI', 'CPI Change (%)', 'Currency-Code', 'Fertility Rate', 'Forested Area (%)', 'Gasoline Price', 'GDP', 'Gross primary education enrollment (%)', 'Gross tertiary education enrollment (%)', 'Infant mortality', 'Largest city', 'Life expectancy', 'Maternal mortality ratio', 'Minimum wage', 'Official language', 'Out of pocket health expenditure', 'Physicians per thousand', 'Population', 'Population: Labor force participation (%)', 'Tax revenue (%)', 'Total tax rate', 'Unemployment rate', 'Urban_population', 'Latitude', 'Longitude']

world_gdp columns:
['country_name', 'indicator_name', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995'

In [7]:
# Clean exchange rates
exrate = exrate.drop(columns=["Unnamed: 0"], errors="ignore")

# Convert date → year
exrate["date"] = pd.to_datetime(exrate["date"], errors="coerce")
exrate["year"] = exrate["date"].dt.year

# Convert exchange rate values
exrate["exchange_rate"] = pd.to_numeric(exrate["value"], errors="coerce")

# Keep only needed columns
exrate = exrate.rename(columns={"Country/Currency": "country"})
exrate = exrate[["country", "currency", "year", "exchange_rate"]]

# Drop bad rows
exrate = exrate.dropna(subset=["year", "exchange_rate"])


/tmp/ipykernel_55/4264589862.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  exrate["date"] = pd.to_datetime(exrate["date"], errors="coerce")


In [8]:
# Clean column names
world_data.columns = world_data.columns.str.strip().str.lower()

# Rename country column
world_data = world_data.rename(columns={"country": "country"})

# Convert key numeric variables if needed
numeric_cols = [
    "gdp", "population", "unemployment rate",
    "cpi", "co2-emissions", "life expectancy"
]

for col in numeric_cols:
    if col in world_data.columns:
        world_data[col] = pd.to_numeric(world_data[col], errors="coerce")

# Drop duplicates
world_data = world_data.drop_duplicates(subset=["country"])

In [9]:
# Identify year columns
gdp_years = [c for c in world_gdp.columns if c.isdigit()]

# Reshape wide → long
world_gdp_long = world_gdp.melt(
    id_vars=["country_name", "indicator_name"],
    value_vars=gdp_years,
    var_name="year",
    value_name="gdp"
)

# Clean data types
world_gdp_long["year"] = world_gdp_long["year"].astype(int)
world_gdp_long["gdp"] = pd.to_numeric(world_gdp_long["gdp"], errors="coerce")

# Drop missing
world_gdp_long = world_gdp_long.dropna(subset=["gdp"])

# Rename for consistency
world_gdp_long = world_gdp_long.rename(columns={
    "country_name": "country"
})

In [10]:
# -----------------------------------
# 1. Standardize column names
# -----------------------------------
for df in [unemp_long, rir, exrate, world_gdp_long]:
    df.columns = df.columns.str.lower().str.strip()

world_data.columns = world_data.columns.str.lower().str.strip()


# -----------------------------------
# 2. Merge PANEL datasets (country + year)
# -----------------------------------
panel = unemp_long.merge(
    rir,
    on=["country", "year"],
    how="inner"
)

panel = panel.merge(
    exrate,
    on=["country", "year"],
    how="left"
)

panel = panel.merge(
    world_gdp_long,
    on=["country", "year"],
    how="left"
)


# -----------------------------------
# 3. Merge COUNTRY‑ONLY dataset (world_data)
# -----------------------------------
panel = panel.merge(
    world_data,
    on="country",
    how="left"
)


# -----------------------------------
# 4. Final cleaning & ordering
# -----------------------------------
panel = panel.sort_values(
    ["country", "year"]
).reset_index(drop=True)

# Remove extreme garbage values (safety)
panel = panel[
    (panel["real_interest_rate"] > -200) &
    (panel["real_interest_rate"] < 200)
]


# -----------------------------------
# 5. Save final dataset (Kaggle output)
# -----------------------------------
panel.to_csv(
    "/kaggle/working/final_panel_dataset.csv",
    index=False
)

panel.head()

,country,country_code_x,year,unemployment_rate,country_code_y,real_interest_rate,currency,exchange_rate,indicator_name,gdp_x,density\n(p/km2),abbreviation,agricultural land( %),land area(km2),armed forces size,birth rate,calling code,capital/major city,co2-emissions,cpi,cpi change (%),currency-code,fertility rate,forested area (%),gasoline price,gdp_y,gross primary education enrollment (%),gross tertiary education enrollment (%),infant mortality,largest city,life expectancy,maternal mortality ratio,minimum wage,official language,out of pocket health expenditure,physicians per thousand,population,population: labor force participation (%),tax revenue (%),total tax rate,unemployment rate,urban_population,latitude,longitude
0,Afghanistan,AFG,2006,11.10,AFG,10.046897,NaN,NaN,Annual GDP growth (percent change),5.4,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,NaN,149.9,2.30%,AFN,4.47,2.10%,$0.70,NaN,104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,NaN,48.90%,9.30%,71.40%,NaN,"9,797,273",33.93911,67.709953
1,Afghanistan,AFG,2007,11.30,AFG,-3.585111,NaN,NaN,Annual GDP growth (percent change),13.3,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,NaN,149.9,2.30%,AFN,4.47,2.10%,$0.70,NaN,104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,NaN,48.90%,9.30%,71.40%,NaN,"9,797,273",33.93911,67.709953
2,Afghanistan,AFG,2008,11.09,AFG,12.557960,NaN,NaN,Annual GDP growth (percent change),3.9,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,NaN,149.9,2.30%,AFN,4.47,2.10%,$0.70,NaN,104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,NaN,48.90%,9.30%,71.40%,NaN,"9,797,273",33.93911,67.709953
3,Afghanistan,AFG,2009,11.31,AFG,17.542929,NaN,NaN,Annual GDP growth (percent change),20.6,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,NaN,149.9,2.30%,AFN,4.47,2.10%,$0.70,NaN,104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,NaN,48.90%,9.30%,71.40%,NaN,"9,797,273",33.93911,67.709953
4,Afghanistan,AFG,2010,11.35,AFG,11.364094,NaN,NaN,Annual GDP growth (percent change),8.4,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,NaN,149.9,2.30%,AFN,4.47,2.10%,$0.70,NaN,104.00%,9.70%,47.9,Kabul,64.5,638.0,$0.43,Pashto,78.40%,0.28,NaN,48.90%,9.30%,71.40%,NaN,"9,797,273",33.93911,67.709953
